# Изотермы H/Pd(110), H/Pd(111) и H/Pt(111) по оцифровке Christmann — исправленная версия

Исходный `plot_bln.ipynb` оставлен без изменений. Исправлено:

1. **Температура**: метки кривых 35…125 — это °C, $T = t + 273.15$ K (было: трактовались как K и экстраполировались к 300–400 K).
2. **Давление**: ось оцифровки в Торр, $P_{\rm Pa} = 133.322\,P_{\rm Torr}$ (было: Па).
3. **Без экстраполяции**: интерполяция только внутри измеренных изотерм и между соседними изотермами по $1/T$; вне данных — `NaN`.
4. Удалён текст про $P_{ref}$, $E_d=96.6$ кДж/моль и $(\Theta/(1-\Theta))^2$ — в коде этой модели не было.

Покрытие по-прежнему $\Theta=\Delta\varphi/\Delta\varphi_{max}$ ($360$ мВ для (110), $180$ мВ для (111)) — это допущение (линейность $\Delta\varphi(\Theta)$), проверяется косвенно в п. 3.

In [ ]:
preparser(False)   # обычная семантика Python/numpy внутри Sage
import os, numpy as np, matplotlib.pyplot as plt

TORR = 133.322368   # Па
R = 8.314462618
DIR = '.' if os.path.exists('Pd_110_p.bln') else 'didger'

def read_bln(fname):
    rows = []
    for line in open(os.path.join(DIR, fname)):
        if '"' in line or not line.strip():
            continue                      # заголовок блока: N,0,"t°C"
        p = line.replace(' ', ',').split(',')
        rows.append([float(x) for x in p[:3]])
    return np.array(rows)

def load(face, dphi_max):
    D = read_bln(f'Pd_{face}_p.bln')
    return dict(P=D[:,0]*TORR, theta=D[:,1]/dphi_max, T=D[:,2]+273.15, face=face)

d110 = load('110', 360.0)
d111 = load('111', 180.0)
for d in (d110, d111):
    print(f"Pd({d['face']}): {len(d['P'])} точек, P = {d['P'].min():.2e}…{d['P'].max():.2e} Па, "
          f"T = {sorted(set(d['T']))} K, Θ = {d['theta'].min():.3f}…{d['theta'].max():.3f}")

ok = all(300 < d['T'].min() and d['T'].max() < 400 and d['P'].max() < 1e-2 for d in (d110, d111))
print('Проверка единиц (T в K 300–400, P < 1e-2 Па):', 'OK' if ok else 'FAIL')

## 1. Проверка пересчёта лог-шкалы давления

В `*_p.bln` давления уже пересчитаны из пиксельной координаты $X$ (`Pd_*.bln`). На лог-оси должно выполняться $\log_{10}P = a + bX$. Невязка показывает точность оцифровки по давлению.

In [ ]:
for face in ('110', '111'):
    X = read_bln(f'Pd_{face}.bln'); Pp = read_bln(f'Pd_{face}_p.bln')
    b, a = np.polyfit(X[:,0], np.log10(Pp[:,0]), 1)
    res = np.abs(np.log10(Pp[:,0]) - (a + b*X[:,0])).max()
    print(f'Pd({face}): {b*1e5:.3f} декад на 1e5 ед. X, max невязка = {res:.4f} декады ->',
          'OK' if res < 0.02 else 'FAIL')

## 2. Интерполяция без экстраполяции

Для изотермы $T_k$: $\Theta_k(\ln P)$ и обратно $\ln P_k(\Theta)$ — линейно, только внутри диапазона точек.
Между соседними изотермами $T_k<T<T_{k+1}$: линейно по $1/T$ (для $\ln P$ это форма Клаузиуса–Клапейрона $\ln P = A - q_{st}/RT$).

In [ ]:
def isotherms(d):
    out = {}
    for T in sorted(set(d['T'])):
        m = d['T'] == T
        o = np.argsort(d['P'][m])
        out[T] = (np.log(d['P'][m][o]), d['theta'][m][o])
    return out

I110, I111 = isotherms(d110), isotherms(d111)

def _on_iso(I, T, x, mode):
    lnP, th = I[T]
    if mode == 'theta':          # x = lnP -> Θ
        return np.interp(x, lnP, th) if lnP[0] <= x <= lnP[-1] else np.nan
    o = np.argsort(th)           # x = Θ -> lnP
    return np.interp(x, th[o], lnP[o]) if th.min() <= x <= th.max() else np.nan

def _across_T(I, T, x, mode):
    Ts = sorted(I)
    if not (Ts[0] <= T <= Ts[-1]): return np.nan
    if T in I: return _on_iso(I, T, x, mode)
    k = np.searchsorted(Ts, T); T1, T2 = Ts[k-1], Ts[k]
    y1, y2 = _on_iso(I, T1, x, mode), _on_iso(I, T2, x, mode)
    w = (1/T - 1/T1)/(1/T2 - 1/T1)
    return (1-w)*y1 + w*y2          # NaN, если хоть одна изотерма вне диапазона

theta_of = lambda I, P, T: _across_T(I, T, np.log(P), 'theta')
P_of     = lambda I, th, T: np.exp(_across_T(I, T, th, 'lnP'))

# самопроверка: на узловой изотерме интерполяция воспроизводит точки
err = max(abs(theta_of(I110, np.exp(l), T) - t) for T,(L,th) in I110.items() for l,t in zip(L,th))
print('Воспроизведение узлов Pd(110):', err, '->', 'OK' if err < 1e-12 else 'FAIL')
print('Вне данных (P=1e-2 Па):', theta_of(I110, 1e-2, 350.0), '-> OK (NaN)' if np.isnan(theta_of(I110,1e-2,350.0)) else 'FAIL')

## 3. Изостерическая теплота — независимая проверка единиц

$q_{st}(\Theta) = -R\,\dfrac{\partial \ln P}{\partial (1/T)}\Big|_\Theta$. Литература (Conrad, Ertl, Latta 1974): начальные теплоты адсорбции ≈ 102 кДж/моль для Pd(110) и ≈ 88 кДж/моль для Pd(111).
Если считать T в кельвинах (35–125 K), получилось бы совсем другое значение, поэтому согласие с литературой подтверждает °C. Небольшой спад $q_{st}$ с ростом $\Theta$ ожидаем.

In [ ]:
def q_st(I, th):
    Ts, L = [], []
    for T in I:
        y = _on_iso(I, T, th, 'lnP')
        if not np.isnan(y): Ts.append(T); L.append(y)
    if len(Ts) < 3: return np.nan, len(Ts)
    return -np.polyfit(1/np.array(Ts), L, 1)[0]*R/1000, len(Ts)

ths = np.linspace(0.1, 0.85, 16)
q110 = [q_st(I110, t)[0] for t in ths]; q111 = [q_st(I111, t)[0] for t in ths]
for face, q, ref in (('110', q110, 102.0), ('111', q111, 88.0)):
    q0 = np.nanmean(q[:4])
    print(f'Pd({face}): q_st(Θ≈0.1–0.25) = {q0:.1f} кДж/моль, литература ≈ {ref} ->',
          'OK' if abs(q0-ref)/ref < 0.15 else 'FAIL')

## 4. Графики

* (a) изотермы обеих граней в единицах Па и K;
* (b) $\Delta\Theta = \Theta_{110}-\Theta_{111}$ при одинаковых $P,T$: каждая грань отдельно в стационаре с газом одного и того же давления;
* (c) $P_{110}/P_{111}$ при одинаковых $\Theta,T$: арифметика по двум независимым изотермам;
* (d) $q_{st}(\Theta)$.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))
cm = plt.cm.viridis
Ts = sorted(I110)
for i, T in enumerate(Ts):
    c = cm(i/(len(Ts)-1))
    ax[0,0].semilogx(np.exp(I110[T][0]), I110[T][1], '-',  color=c, label=f'{T-273.15:.0f} °C')
    ax[0,0].semilogx(np.exp(I111[T][0]), I111[T][1], '--', color=c)
ax[0,0].set(xlabel='P, Па', ylabel='Θ', title='(a) изотермы: — Pd(110), -- Pd(111)'); ax[0,0].legend(fontsize=8)

Pg = np.logspace(np.log10(6e-7), np.log10(8e-4), 120)
for i, T in enumerate(Ts):
    dth = [theta_of(I110,p,T) - theta_of(I111,p,T) for p in Pg]
    ax[0,1].semilogx(Pg, dth, color=cm(i/(len(Ts)-1)), label=f'{T-273.15:.0f} °C')
ax[0,1].axhline(0, color='k', lw=.5)
ax[0,1].set(xlabel='P, Па', ylabel='Θ₁₁₀ − Θ₁₁₁', title='(b) ΔΘ при равных P, T'); ax[0,1].legend(fontsize=8)

thg = np.linspace(0.02, 0.97, 120)
for i, T in enumerate(Ts):
    r = [P_of(I110,t,T)/P_of(I111,t,T) for t in thg]
    ax[1,0].semilogy(thg, r, color=cm(i/(len(Ts)-1)), label=f'{T-273.15:.0f} °C')
ax[1,0].axhline(1, color='k', lw=.5)
ax[1,0].set(xlabel='Θ', ylabel='P₁₁₀ / P₁₁₁', title='(c) отношение давлений при равных Θ, T'); ax[1,0].legend(fontsize=8)

ax[1,1].plot(ths, q110, 'o-', label='Pd(110)'); ax[1,1].plot(ths, q111, 's--', label='Pd(111)')
ax[1,1].set(xlabel='Θ', ylabel='q_st, кДж/моль', title='(d) изостерическая теплота'); ax[1,1].legend()
for a in ax.flat: a.grid(True, which='both', ls=':', alpha=.6)
plt.tight_layout(); plt.savefig('plot_bln_v2.png', dpi=110); plt.show()

## 6. Pt(111)

Файлы `Pt_111*.bln`: 16 изотерм, 401 точка. В отличие от Pd, здесь:

* **ось Y — уже покрытие $\Theta$** (0…0.93), а не $\Delta\varphi$ в мВ, поэтому делить на $\Delta\varphi_{max}$ не нужно;
* **метки кривых 165…366 — это кельвины**, а не °C. Проверка через $q_{st}$ ниже: при T в K получается ≈ 37–42 кДж/моль, это согласуется с литературой (Christmann, Ertl, Comsa 1976: ≈ 39 кДж/моль для H/Pt(111)). Если считать их °C, вышло бы 130–200 кДж/моль, и $q_{st}$ сильно *рос* бы с $\Theta$, что для H/Pt физически неверно;
* давление, как и раньше, в Торр (1e-8…1e-5).

Интервалы температур Pt и Pd перекрываются только на 308–366 K.

In [ ]:
def load_pt():
    D = read_bln('Pt_111_p.bln')
    return dict(P=D[:,0]*TORR, theta=D[:,1], T=D[:,2], face='Pt111')   # T уже в K, Y уже Θ
dPt = load_pt(); IPt = isotherms(dPt)
print(f"Pt(111): {len(dPt['P'])} точек, {len(IPt)} изотерм, T = {min(IPt)}…{max(IPt)} K, "
      f"P = {dPt['P'].min():.2e}…{dPt['P'].max():.2e} Па")

X = read_bln('Pt_111.bln'); Pp = read_bln('Pt_111_p.bln')
b, a = np.polyfit(X[:,0], np.log10(Pp[:,0]), 1)
res = np.abs(np.log10(Pp[:,0]) - (a + b*X[:,0])).max()
print(f'Лог-шкала: max невязка = {res:.4f} декады ->', 'OK' if res < 0.02 else 'FAIL')

mono = all(np.all(np.diff(th) > -0.01) for (l, th) in IPt.values())
print('Θ(P) монотонна на каждой изотерме:', 'OK' if mono else 'FAIL')

def q_alt(I, th, dT):     # то же, но со сдвигом шкалы T (для проверки гипотезы °C)
    J = {T+dT: v for T, v in I.items()}
    return q_st(J, th)[0]
thsPt = np.linspace(0.05, 0.6, 12)
qPt  = [q_st(IPt, t)[0] for t in thsPt]
qPtC = [q_alt(IPt, t, 273.15) for t in thsPt]
q0 = np.nanmean(qPt[:4])
print(f'q_st(Θ≈0.05–0.2), T в K : {q0:.1f} кДж/моль, литература ≈ 39 ->', 'OK' if abs(q0-39)/39 < 0.15 else 'FAIL')
print(f'q_st, если считать T в °C : {np.nanmean(qPtC[:4]):.0f}…{np.nanmax(qPtC):.0f} кДж/моль -> отвергнуто')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
TsPt = sorted(IPt)
for i, T in enumerate(TsPt):
    ax[0].semilogx(np.exp(IPt[T][0]), IPt[T][1], color=cm(i/(len(TsPt)-1)), label=f'{T:.0f} K')
ax[0].set(xlabel='P, Па', ylabel='Θ', title='(e) изотермы H/Pt(111)'); ax[0].legend(fontsize=7, ncol=2)

ax[1].plot(thsPt, qPt, 'o-', color='C2', label='Pt(111)')
ax[1].plot(ths, q110, 'o-', label='Pd(110)'); ax[1].plot(ths, q111, 's--', label='Pd(111)')
ax[1].set(xlabel='Θ', ylabel='q_st, кДж/моль', title='(f) изостерическая теплота'); ax[1].legend()

# ΔΘ = Θ_Pd(111) − Θ_Pt(111) при одинаковых P,T в общей области
for T in (308.15, 323.15, 338.15, 353.15):
    dth = [theta_of(I111, p, T) - theta_of(IPt, p, T) for p in Pg]
    ax[2].semilogx(Pg, dth, label=f'{T:.0f} K')
ax[2].set(xlabel='P, Па', ylabel='Θ_Pd(111) − Θ_Pt(111)', title='(g) ΔΘ Pd/Pt при равных P, T'); ax[2].legend()
for a in ax: a.grid(True, which='both', ls=':', alpha=.6)
plt.tight_layout(); plt.savefig('plot_bln_v2_Pt.png', dpi=110); plt.show()

**Вывод по Pt(111).**

*Из данных.* $q_{st}$ ≈ 40 кДж/моль против 90–100 для Pd, то есть водород на Pt связан примерно в 2.3 раза слабее. При одинаковых $P,T$ платина покрыта намного меньше (g).

*Чего данные не говорят.* Изотермы относятся к отдельным кристаллам в контакте с газом, металлического объёма между гранями в них нет. Создаст ли разность $\Theta$ поток через мембрану Pd|Pt, по этим данным решить нельзя. Утверждение «потока нет» верно только при допущении детального равновесия: тогда цепочка газ → поверхность → объём у обеих сторон одинакова. Это допущение, а не результат.

## 5. Выводы

Разделено на три уровня: что следует из данных; что следует только при допущении детального равновесия; что остаётся открытым.

**А. Из данных (проверено в ячейках выше)**

* Все величины лежат внутри измеренной области: Pd — $T = 308\ldots398$ K, $P \approx 5\cdot10^{-7}\ldots8\cdot10^{-4}$ Па.
* $q_{st}$ согласуется с литературой, значит единицы (°C и Торр для Pd, K и Торр для Pt) выбраны верно.
* (b) В газе с одинаковыми $P,T$ стационарные покрытия граней **разные**.
* (c) Для одинаковых $\Theta$ стационарные давления над гранями отличаются в ≈1…13 раз. Это сопоставление двух независимых опытов. Что такую пару состояний можно реализовать на двух сторонах одной мембраны, оно не означает.

**Б. Только при допущении детального равновесия (постулат, здесь не проверялся)**

* Если в каждом звене газ ⇄ поверхность ⇄ объём выполняется детальное равновесие, у обеих граней один и тот же закон Сивертса $c(P)$. Тогда при $J=0$ давления по обе стороны равны, и ни (b), ни (c) движущей силы не создают.
* Тот же постулат лежит в основе утверждения, что состояние (c) нужно навязывать извне. **Из изотерм он не следует.**

**В. Открыто — решается кинетикой, а не изотермами**

* Нужна модель мембраны с явными константами: адсорбция и десорбция на каждой грани, обмен поверхность ⇄ объём, диффузия в объёме. Из неё считается стационарное $P_B/P_A$ при $J=0$, без условия на $\mu_H$.
* Проверка: условия Вегшайдера (произведение констант по замкнутому циклу газ → грань A → объём → грань B → газ должно равняться 1) на **измеренных** константах. Если оно ≠ 1 с запасом больше погрешностей, это количественное свидетельство против постулата; если = 1 — в его пользу.
* Экспериментальные критерии, которые нужно сверить с источниками: зависит ли растворимость H в Pd от грани, через которую идёт насыщение; есть ли у Лифшица перепад давлений на мембране при чистом H₂ с обеих сторон.

In [ ]:
# Проверка согласованности интерполяции: P -> Θ -> P на каждой грани.
# Это проверка численного метода, а НЕ утверждения о μ_H или о потоке через мембрану.
T, P = 353.15, 1e-4
print(f'T={T} K, P={P} Па: Θ110={theta_of(I110,P,T):.3f}, Θ111={theta_of(I111,P,T):.3f}; '
      f'P обратно: {P_of(I110,theta_of(I110,P,T),T):.3e}, {P_of(I111,theta_of(I111,P,T),T):.3e}')
ok = abs(P_of(I110,theta_of(I110,P,T),T)/P-1) < 0.02 and abs(P_of(I111,theta_of(I111,P,T),T)/P-1) < 0.02
print('Обратимость P->Θ->P:', 'OK' if ok else 'FAIL')